# ST 554 Project 2
*By: Cass Crews*

In this notebook, we will explore the capabilities of the `PySpark` API. More specifically, we will demonstrate the functionality of the `PySpark.sql` and `PySpark.pandas` modules, two key modules that bridge the gap between Python's functionalities and the big data capabilities of Apache Spark. 

In Part I, we will use a custom class, specifically built for this notebook and stored in the same repository, to conduct basic data validation on the SQL-style dataframes central to `PySpark.sql`. 

In Part II, we will complete the same data manipulation process using the `PySpark.sql` and `PySpark.pandas` modules, highlighting their similarities and differences. 

Before we get started, we need to read in the modules and sub-modules we'll use throughout the notebook. We also need to initialize a Spark session. 

In [4]:
# Importing key modules and sub-modules, and initializing a Spark session
import numpy as np
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

## Part I

While `PySpark.sql` is incredibly useful for accessing and manipulating databases, some steps of a standard exploratory data analysis (EDA) can be more tedious than they are when working in R's tidyverse or when utilizing the `pandas` module. Thus, Part I explores a custom class, `SparkDataCheck`, created specifically for this notebook to highlight the abilities to use classes and associated methods to simplify common EDA steps. The source script for `SparkDataCheck` can be found [here](https://github.com/jccrews256/ST-554-Project-2/blob/main/SparkDataCheck.py). 

`SparkDataCheck` is designed to be initialized in three convenient ways, with each resulting in a `.df` attribute that stores a dataset in a `PySpark.sql` dataframe:

- `.__init__()`: This initializing method directly takes in a `PySpark.sql` dataframe
- `.from_csv()`: This class method creates an instance of our custom class while reading in the data from a csv file
    - In addition to the file path, we need to provide the Spark session information
- `.from_pddf()`: This class method creates an instance of our custom class from a `pandas` dataframe
    - In addition to the dataframe name, we need to provide the Spark session information

Of course, to utilize any of these three methods, we need to make the class accessible in our environment. Let's do that now. 

*Note: The `air.csv` and `SparkDataCheck.py` files will need to be available in the reader's environment to run all code chunks in Part I.*

In [5]:
from SparkDataCheck import *

Before we begin exploring objects of the `SparkDataCheck` class, let's print the docstring to fully understand what the class offers. 

In [6]:
# Printing class docstring
print(SparkDataCheck.__doc__)


    A class designed to assist the user in validating a PySpark.sql dataframe,
    stored as the attribute df. 
    
    In addition to using __init__ to directly initialize the class with a 
    pre-existing PySpark.sql dataframe, there are two convenient class methods
    to create an object of class SparkDataCheck:
        - from_csv: a class method that takes in the Spark session (spark) and
        a file path to a csv data file (csv_path) as inputs to create the object,
        storing the dataset as a PySpark.sql dataframe in df
        - from_pddf: a class method that takes in the Spark session (spark) and
        a pandas dataframe (pd_df) as inputs to create the object, storing the 
        dataset as a PySpark.sql dataframe in df
        
    Note that these two class methods will not produce identical results because
    pandas and PySpark.sql exhibit different csv-to-dataframe behaviors. 
    
    Methods:
        - in_range: a method that takes in a numeric column (colum

The docstring notes the three ways of initializing the class, as well as five methods. After we create a `SparkDataCheck` object, we will thoroughly explore each method. 

#### Initializing Class Using `from_csv`

Let's utilize the `from_csv()` class method to initialize our custom class while reading in a csv file of air pollution readings. This dataset was developed to test the efficacy of using low-cost chemical sensors to estimate true pollutant concentrations in situations where pollution monitoring systems are cost-prohibitive. Thus, the data contains hourly true concentrations and sensor readings for multiple pollutants in a single location. We will not explore the context further as our focus is on exploring the custom class. However, interested readers can access the data [here](https://archive.ics.uci.edu/dataset/360/air+quality) and learn more about the study [here](https://www.sciencedirect.com/science/article/abs/pii/S0925400507007691). 

In [7]:
# Creating a SparkDataCheck object containing the air quality data
test_data_sql = SparkDataCheck.from_csv(spark, "air.csv")

Let's print the first 25 rows of the dataset to gain a better understanding of the data structure. 

In [8]:
# Extracting the first 25 rows of the data
test_data_sql.df.show(25)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-22 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-22 21:00:00|   2.2|       137

26/03/22 11:10:01 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-jccrews@ncsu.edu/air.csv


Note the warning, which indicates the first column in the file does not have a name. This is simply the row index, so the warning is not concerning. In the next code chunk, we'll adjust the settings of our Spark session so that warnings are no longer printed.

Another interesting note is that the `Time` values are assigned the date corresponding to when the code is run. This is because `PySpark` coerces times to time stamps and uses the date of run as a placeholder date.

The rows printed above indicate an idiosyncrasy of the dataset that we will need to handle before we explore the methods of our custom class: missing values are coded -200.  Let's loop over the atmospheric measurements (`CO(GT)` through `AH`) to convert each `-200` to `NULL`. In the process, we will update the variable names because the "." in some of the names will be problematic for `PySpark`. 

In [9]:
# Suppressing future warnings but not errors
spark.sparkContext.setLogLevel("ERROR")

# Specifying clean variable names
clean_names = ['Index', 'Date', 'Time', 'CO_true', 'CO_sensor', 'NMHC_true', 'C6H6_true', 
               'NMHC_sensor', 'NOx_true', 'NOx_sensor', 'NO2_true', 'NO2_sensor', 'O3_sensor', 
               'Temp', 'Rel_Humid', 'Abs_Humid']

# Updating names and converting the missing values (labeled -200) to NULL
for i in range(len(test_data_sql.df.columns)):
    # Updating names
    test_data_sql.df = test_data_sql.df.withColumnRenamed(test_data_sql.df.columns[i], clean_names[i])
    
    # Converting missing values for the measurements
    if i >= 3:
        test_data_sql.df = test_data_sql.df.withColumn(clean_names[i], F.when(test_data_sql.df[clean_names[i]] == -200, None).otherwise(test_data_sql.df[clean_names[i]]))        

Let's print out the first 25 rows again to ensure our updates worked. 

In [10]:
# Extracting the first 25 rows of the data
test_data_sql.df.show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+
|    0|3/10/2004|2026-03-22 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|
|    1|3/10/2004|2026-03-22 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|
|    2|3/10/2004|2026-03-22 20:00:00|    2.2|     1402|       88|      9.0|        939|     131|      1140|     114|      1555|     1074|11.9|     54.0|   0.7502|
|    3|3/10/2004|2026-

We've now made all the updates we need to explore `SparkDataCheck`'s methods. We'll be doing so with variables names that are easier to interpret for readers who are familiar with chemical symbols. For example, the `CO_true` column contains the true carbon monoxide concentrations. 

#### Testing the `in_range()` Method

Let's start with the `in_range()` method, which adds a new column to the dataframe indicating whether the values of a numeric column in the dataset (`column`) are between the `lower` and `upper` inputs, inclusive. At least one of these bounding inputs must be provided. 

For our first use of `in_range()`, let's identify which true carbon monoxide concentrations are between 1 and 3. 

In [11]:
# Adding new boolean column indicating whether CO_true between 1 and 3
test_data_sql.in_range(column = "CO_true", lower = 1, upper = 3).show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+
|    0|3/10/2004|2026-03-22 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|          true|
|    1|3/10/2004|2026-03-22 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|          true|
|    2|3/10/2004|2026-03-22 20:00:00|    2.2|     1402|       88|      9.0|        939|     131|      1140|   

When we look at the 25 observations we are becoming rather familiar with, we now see a column of "true" and "false" values. I must say, this new column has a great name! 

One thing we should note: when the input column is `NULL`, the new column is also `NULL`. This will be convenient if we want to count true and false values without worrying about including missing values. 

Next, let's generate a new column that indicates when `CO_true` is greater than or equal to 1, with no upper bound. 

In [12]:
# Adding new boolean column indicating whether CO_true above 1
test_data_sql.in_range(column = "CO_true", lower = 1).show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|CO_true_above_1|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+
|    0|3/10/2004|2026-03-22 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|          true|           true|
|    1|3/10/2004|2026-03-22 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|          true|           true|
|    2|3/10/2004|2026-03-22 20

Comparing the two Boolean columns we have created so far, it seems a lower bound of 1 is more restrictive than an upper bound of 3 -- at least for the first 25 observations. 

Let's create another column that indicates whether the concentrations are below 3, isolating these seemingly rare occurrences above 3; we'll subset the columns we `show` so that the printed table is not too crowded. I suspect the reader is wondering how these columns will be used. Do not fret, as another method for this class can be used to count "true" and "false" values. 

In [13]:
# Adding new boolean column indicating whether CO_true below 3
test_data_sql.in_range(column = "CO_true", upper = 3).select("Date", "Time", "CO_true", "CO_true_in_1_3", "CO_true_above_1", "CO_true_below_3").show(25)

+---------+-------------------+-------+--------------+---------------+---------------+
|     Date|               Time|CO_true|CO_true_in_1_3|CO_true_above_1|CO_true_below_3|
+---------+-------------------+-------+--------------+---------------+---------------+
|3/10/2004|2026-03-22 18:00:00|    2.6|          true|           true|           true|
|3/10/2004|2026-03-22 19:00:00|    2.0|          true|           true|           true|
|3/10/2004|2026-03-22 20:00:00|    2.2|          true|           true|           true|
|3/10/2004|2026-03-22 21:00:00|    2.2|          true|           true|           true|
|3/10/2004|2026-03-22 22:00:00|    1.6|          true|           true|           true|
|3/10/2004|2026-03-22 23:00:00|    1.2|          true|           true|           true|
|3/11/2004|2026-03-22 00:00:00|    1.2|          true|           true|           true|
|3/11/2004|2026-03-22 01:00:00|    1.0|          true|           true|           true|
|3/11/2004|2026-03-22 02:00:00|    0.9|    

We now have three boolean columns effectively splitting the empirical distribution of true carbon monoxide concentration into fixed regions. We will soon use these columns to understand the likelihoods of falling into each region.  

Before moving onto the next method, let's see what happens when we pass a string column to `in_range`. 

In [14]:
# Testing case when input column is not numeric
test_data_sql.in_range(column = "Date", lower = 1).drop("Index").show(25)

Column must be a numeric type
+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+---------------+
|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|CO_true_above_1|CO_true_below_3|
+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+---------------+
|3/10/2004|2026-03-22 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|          true|           true|           true|
|3/10/2004|2026-03-22 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.725

This is useful functionality: the method returns a message indicating the column must be a numeric type, but it also returns the unmodified dataframe so that we can proceed in chaining operations with no issues.

#### Testing the `in_set` Method

In many ways, the `in_set` method is the categorical variable analog to `in_range`. It returns a Boolean column indicating whether the value of a `string` column (`column`) is within a set of `levels`. To capture the context of the new column, the user can optionally provide a `tag` that will be incorporated into the column name. 

Let's test the `in_set` method by indicating when the date is "3/11/2004"; we'll set `tag` to "mar_11" for future reference. As our dataframe is becoming rather wide, we will again subset the columns printed. 

In [15]:
# Adding new boolean column indicating when Date is 3/11/2004
test_data_sql.in_set(column = "Date", levels = "3/11/2004", tag = "mar_11") \
    .select("Date", "Time", "Date_in_set_mar_11").show(25)

+---------+-------------------+------------------+
|     Date|               Time|Date_in_set_mar_11|
+---------+-------------------+------------------+
|3/10/2004|2026-03-22 18:00:00|             false|
|3/10/2004|2026-03-22 19:00:00|             false|
|3/10/2004|2026-03-22 20:00:00|             false|
|3/10/2004|2026-03-22 21:00:00|             false|
|3/10/2004|2026-03-22 22:00:00|             false|
|3/10/2004|2026-03-22 23:00:00|             false|
|3/11/2004|2026-03-22 00:00:00|              true|
|3/11/2004|2026-03-22 01:00:00|              true|
|3/11/2004|2026-03-22 02:00:00|              true|
|3/11/2004|2026-03-22 03:00:00|              true|
|3/11/2004|2026-03-22 04:00:00|              true|
|3/11/2004|2026-03-22 05:00:00|              true|
|3/11/2004|2026-03-22 06:00:00|              true|
|3/11/2004|2026-03-22 07:00:00|              true|
|3/11/2004|2026-03-22 08:00:00|              true|
|3/11/2004|2026-03-22 09:00:00|              true|
|3/11/2004|2026-03-22 10:00:00|

This is another useful method when we want to identify observations with certain characteristics. 

If we want to identify all observations, corresponding to 6am, we can pull the hour from `Time` and use that in conjunction with `in_set`. 

In [16]:
# Extracting the hour as a new column
test_data_sql.df = test_data_sql.df.withColumn("Hour", F.hour("Time").cast("string"))

# Adding new boolean column indicating when the hour is 6
test_data_sql.in_set(column = "Hour", levels = "6", tag = "6am") \
    .select("Date", "Time", "Hour_in_set_6am").show(25)

+---------+-------------------+---------------+
|     Date|               Time|Hour_in_set_6am|
+---------+-------------------+---------------+
|3/10/2004|2026-03-22 18:00:00|          false|
|3/10/2004|2026-03-22 19:00:00|          false|
|3/10/2004|2026-03-22 20:00:00|          false|
|3/10/2004|2026-03-22 21:00:00|          false|
|3/10/2004|2026-03-22 22:00:00|          false|
|3/10/2004|2026-03-22 23:00:00|          false|
|3/11/2004|2026-03-22 00:00:00|          false|
|3/11/2004|2026-03-22 01:00:00|          false|
|3/11/2004|2026-03-22 02:00:00|          false|
|3/11/2004|2026-03-22 03:00:00|          false|
|3/11/2004|2026-03-22 04:00:00|          false|
|3/11/2004|2026-03-22 05:00:00|          false|
|3/11/2004|2026-03-22 06:00:00|           true|
|3/11/2004|2026-03-22 07:00:00|          false|
|3/11/2004|2026-03-22 08:00:00|          false|
|3/11/2004|2026-03-22 09:00:00|          false|
|3/11/2004|2026-03-22 10:00:00|          false|
|3/11/2004|2026-03-22 11:00:00|         

We can now easily identify all measurements taken at 6am! It will be interesting to see how pollution at this time compares to pollution in the other 23 hours of the day. 

Let's see what happens when we provide a numeric column. We'll drop a few measurements so the printed table isn't wider than the notebook itself. 

In [17]:
# Testing behavior when column is numeric
test_data_sql.in_set(column = "CO_true", levels = "5", tag = "bad_input") \
    .drop('Index', 'CO_sensor', 'NMHC_true', 'C6H6_true', 
               'NMHC_sensor', 'NOx_true', 'NOx_sensor', 
          'NO2_true', 'NO2_sensor', 'O3_sensor').show(25)

Column must be a string type
+---------+-------------------+-------+----+---------+---------+--------------+---------------+---------------+------------------+----+---------------+
|     Date|               Time|CO_true|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|CO_true_above_1|CO_true_below_3|Date_in_set_mar_11|Hour|Hour_in_set_6am|
+---------+-------------------+-------+----+---------+---------+--------------+---------------+---------------+------------------+----+---------------+
|3/10/2004|2026-03-22 18:00:00|    2.6|13.6|     48.9|   0.7578|          true|           true|           true|             false|  18|          false|
|3/10/2004|2026-03-22 19:00:00|    2.0|13.3|     47.7|   0.7255|          true|           true|           true|             false|  19|          false|
|3/10/2004|2026-03-22 20:00:00|    2.2|11.9|     54.0|   0.7502|          true|           true|           true|             false|  20|          false|
|3/10/2004|2026-03-22 21:00:00|    2.2|11.0|     60.0|   0.

Similar to `in_range`, `in_set` takes improper column inputs in stride, simply printing the message "Column must be a string type" and returning the unmodified dataframe. 

If we want to identify the `NULL` values in the `CO_true` column, we can try to use `in_set` to flag "NULL" in a string-cast version of `CO_true_in_1_3`. 

In [18]:
# Casting CO_true_in_1_3 to string
test_data_sql.df = test_data_sql.df.withColumn("CO_true_in_1_3_str", F.col("CO_true_in_1_3").cast("string"))

# Adding new boolean column indicating when this new string is NULL
test_data_sql.in_set(column = "CO_true_in_1_3_str", levels = "NULL", tag = "null") \
    .select("Date", "Time", "CO_true_in_1_3_str", "CO_true_in_1_3_str_in_set_null").show(25)

+---------+-------------------+------------------+------------------------------+
|     Date|               Time|CO_true_in_1_3_str|CO_true_in_1_3_str_in_set_null|
+---------+-------------------+------------------+------------------------------+
|3/10/2004|2026-03-22 18:00:00|              true|                         false|
|3/10/2004|2026-03-22 19:00:00|              true|                         false|
|3/10/2004|2026-03-22 20:00:00|              true|                         false|
|3/10/2004|2026-03-22 21:00:00|              true|                         false|
|3/10/2004|2026-03-22 22:00:00|              true|                         false|
|3/10/2004|2026-03-22 23:00:00|              true|                         false|
|3/11/2004|2026-03-22 00:00:00|              true|                         false|
|3/11/2004|2026-03-22 01:00:00|              true|                         false|
|3/11/2004|2026-03-22 02:00:00|             false|                         false|
|3/11/2004|2026-

That did not accomplish our goal.... It seems we need another tool in the toolbox. 

#### Testing the `is_null` Method

The `is_null` method was created for the explicit purpose of flagging `NULL` values in a `column`. Let's use that on the original `CO_true_in_1_3`. 

In [19]:
# Adding a boolean column indicating when CO_true_in_1_3 is NULL
test_data_sql.is_null(column = "CO_true_in_1_3").select("Date", "Time", 
                                               "CO_true_in_1_3", 
                                               "CO_true_in_1_3_is_null").show(25)

+---------+-------------------+--------------+----------------------+
|     Date|               Time|CO_true_in_1_3|CO_true_in_1_3_is_null|
+---------+-------------------+--------------+----------------------+
|3/10/2004|2026-03-22 18:00:00|          true|                 false|
|3/10/2004|2026-03-22 19:00:00|          true|                 false|
|3/10/2004|2026-03-22 20:00:00|          true|                 false|
|3/10/2004|2026-03-22 21:00:00|          true|                 false|
|3/10/2004|2026-03-22 22:00:00|          true|                 false|
|3/10/2004|2026-03-22 23:00:00|          true|                 false|
|3/11/2004|2026-03-22 00:00:00|          true|                 false|
|3/11/2004|2026-03-22 01:00:00|          true|                 false|
|3/11/2004|2026-03-22 02:00:00|         false|                 false|
|3/11/2004|2026-03-22 03:00:00|         false|                 false|
|3/11/2004|2026-03-22 04:00:00|          NULL|                  true|
|3/11/2004|2026-03-2

Because `is_null` can handle any type of column, we can obtain the same result by passing `CO_true` directly. 

In [20]:
# Adding a column indicating when CO_true is NULL
test_data_sql.is_null(column = "CO_true").select("Date", "Time", "CO_true", 
                                        "CO_true_in_1_3", "CO_true_in_1_3_is_null", 
                                        "CO_true_is_null").show(25)

+---------+-------------------+-------+--------------+----------------------+---------------+
|     Date|               Time|CO_true|CO_true_in_1_3|CO_true_in_1_3_is_null|CO_true_is_null|
+---------+-------------------+-------+--------------+----------------------+---------------+
|3/10/2004|2026-03-22 18:00:00|    2.6|          true|                 false|          false|
|3/10/2004|2026-03-22 19:00:00|    2.0|          true|                 false|          false|
|3/10/2004|2026-03-22 20:00:00|    2.2|          true|                 false|          false|
|3/10/2004|2026-03-22 21:00:00|    2.2|          true|                 false|          false|
|3/10/2004|2026-03-22 22:00:00|    1.6|          true|                 false|          false|
|3/10/2004|2026-03-22 23:00:00|    1.2|          true|                 false|          false|
|3/11/2004|2026-03-22 00:00:00|    1.2|          true|                 false|          false|
|3/11/2004|2026-03-22 01:00:00|    1.0|          true|      

These two `NULL`-identifying Boolean columns should be identical, and that seems to be the case based on the first 25 rows. 

#### Testing the `calc_min_max` Method

How can we extract information that explicitly validates the data, rather than simply generating Boolean columns? One option is the `calc_min_max` method, which returns the min and max of one or all numeric columns either unconditionally or conditional on a `groupby` input. The `groupby` input is a single column that should be categorical in nature to prevent excessively fine grouping, but there are no limitations on the formal column type. 

To start, let's extract the unconditional minimums and maximums for each numeric column, which can be done by not providing any additional information to the method. 

In [21]:
# Extracting the unconditional min and max of all numeric columns
test_data_sql.calc_min_max()

,min(Index),max(Index),min(CO_true),max(CO_true),min(CO_sensor),max(CO_sensor),min(NMHC_true),max(NMHC_true),min(C6H6_true),max(C6H6_true),...,min(NO2_sensor),max(NO2_sensor),min(O3_sensor),max(O3_sensor),min(Temp),max(Temp),min(Rel_Humid),max(Rel_Humid),min(Abs_Humid),max(Abs_Humid)
0,0,9356,0.1,11.9,647,2040,7,1189,0.1,63.7,...,551,2775,221,2523,-1.9,44.6,9.2,88.7,0.1847,2.231


Three notes: First, the results are conveniently returned as a `pandas` dataframe, which is reasonable for summary statistics; `PySpark` dataframes are unnecessary here. Second, `Index` is technically numeric, but its min and max only characterize the number of rows (9356+1=9357). Third, this is an overwhelming amount of information. 

Let's generate the minimums and maximums for each numeric variable across the hours of the day simply to demonstrate the ability to generate conditional endpoints for each empirical numeric distribution in this dataset, then transition to single-column evaluations. `calc_min_max` automatically sorts the dataframe by the `groupby` column, so let's convert `Hour` to an integer to ensure the sorting is intuitive. 

In [22]:
# Casting Hour to numeric for intuitive sorting
test_data_sql.df = test_data_sql.df.withColumn("Hour_int", test_data_sql.df["Hour"].cast("int"))

# Extracting the conditional min and max of all numeric columns across Hour_int
test_data_sql.calc_min_max(groupby = "Hour_int")

,Hour_int,min(Index),max(Index),min(CO_true),max(CO_true),min(CO_sensor),max(CO_sensor),min(NMHC_true),max(NMHC_true),min(C6H6_true),...,min(O3_sensor),max(O3_sensor),min(Temp),max(Temp),min(Rel_Humid),max(Rel_Humid),min(Abs_Humid),max(Abs_Humid),min(Hour_int),max(Hour_int)
22,0,6,9342,0.1,5.9,683,1712,31,275,1.0,...,322,2411,0.2,29.8,14.0,84.1,0.1910,2.1395,0,0
2,1,7,9343,0.1,5.6,692,1751,27,251,0.6,...,307,2519,0.0,28.4,14.5,83.9,0.1847,2.1059,1,1
21,2,8,9344,0.1,5.5,679,1574,23,173,0.4,...,268,1967,-0.1,28.3,17.0,84.6,0.1975,2.1247,2,2
6,3,9,9345,0.1,5.2,676,1458,19,132,0.2,...,227,1960,-1.1,28.8,19.2,86.6,0.2379,2.1074,3,3
13,4,10,9346,0.1,2.8,681,1515,9,96,0.2,...,221,1847,-1.4,28.9,22.5,86.6,0.2326,2.1010,4,4
8,5,11,9347,0.1,2.9,678,1491,7,93,0.1,...,225,1792,-1.3,27.5,23.1,85.7,0.2347,2.1195,5,5
4,6,12,9348,0.1,3.5,655,1453,16,179,0.1,...,232,1680,-1.2,27.5,24.7,85.5,0.2404,2.1170,6,6
16,7,13,9349,0.1,5.7,649,1689,27,754,0.2,...,261,1869,-1.3,27.6,30.9,85.3,0.2478,2.1008,7,7
14,8,14,9350,0.1,7.3,647,1797,36,1129,0.3,...,274,2176,-1.9,29.0,24.0,84.5,0.2478,2.1719,8,8
11,9,15,9351,0.1,8.1,667,1961,35,798,0.4,...,291,2346,-0.5,31.0,20.0,83.9,0.2388,2.1362,9,9


It's great to know this is possible for datasets with fewer numeric variables, but the table provides too much information in this case. 

Let's generate the hour-dependent minimums and maximums for `CO_true` only by passing this variable to `column`. 

In [23]:
# Extracting the conditional min and max of CO_true across Hour_int
test_data_sql.calc_min_max(column = "CO_true", groupby = "Hour_int")

,Hour_int,min(CO_true),max(CO_true)
22,0,0.1,5.9
2,1,0.1,5.6
21,2,0.1,5.5
6,3,0.1,5.2
13,4,0.1,2.8
8,5,0.1,2.9
4,6,0.1,3.5
16,7,0.1,5.7
14,8,0.1,7.3
11,9,0.1,8.1


These results are fascinating; the min values are similar across the day, but the max values show an interesting pattern. In particular, carbon monoxide levels climb to a local peak as people start their days, then hit a higher peak in the evening. The divergence in trends between min and max values may indicate that a few entire days exhibit low polluting activity.

To explore this further, let's use `in_set` to isolate a random winter Sunday (February 6, 2005) and compare its min and max carbon monoxide levels to the min and max for all other days. 

In [24]:
# Adding new boolean column indicating when Date is 2/6/2005
test_data_sql.in_set("Date", "2/6/2005", tag = "feb_6")

# Extracting the conditional min and max of CO_true across Hour_int
test_data_sql.calc_min_max(column = "CO_true", groupby = "Date_in_set_feb_6")

,Date_in_set_feb_6,min(CO_true),max(CO_true)
1,False,0.1,11.9
0,True,0.6,2.0


Indeed, carbon monoxide levels are relatively low across February 6th. 

As a final test, let's see what happens if we provide a string column to `column`. 

In [25]:
# Testing a string column input to column
test_data_sql.calc_min_max(column = "Hour")

Column must be a numeric type


This is convenient: the method simply prints a message indicating `column` must be numeric. If it instead threw an error, an attempt to run all code chunks in this notebook would stop at this chunk. 

#### Testing the `count_combos` Method

The first three methods we explored create Boolean columns based on some condition. It would be helpful to know the breakdown between true's and false's. `count_combos` can help us do just that. This method takes in up to two string columns (`column1` and `column2`). If only `column1` is input, the method returns a `pandas` dataframe with counts for each level of the column. If two columns are input, the method returns a dataframe with counts for each combination of the two columns' levels. 

Let's use the method to count the number of observations with `CO_true` values between 1 and 3. To do so, we can use the string-cast version of `CO_true_in_1_3` we already created. 

In [26]:
# Counting observations where CO_true between 1 and 3 (and not)
test_data_sql.count_combos(column1 = "CO_true_in_1_3_str")

,CO_true_in_1_3_str,count
0,None,1683
1,false,3271
2,true,4403


It seems the carbon monoxide concentration is more likely to be between 1 and 3 than not. Also, nearly 1,700 observations are missing a value for `CO_true`. I wonder if these are due to power outages, monitoring station maintenance, or some other issue.  

We can combine `CO_true_in_1_3_str` with a string-cast version of `CO_true_above_1` to understand how often the carbon monoxide concentration is below 1, between 1 and 3, and above 3. 

In [27]:
# Casting CO_true_above_1 to string
test_data_sql.df = test_data_sql.df.withColumn("CO_true_above_1_str", F.col("CO_true_above_1").cast("string"))

# Counting observations for combinations of levels for CO_true_in_1_3 and CO_true_above_1
test_data_sql.count_combos(column1 = "CO_true_in_1_3_str", column2 = "CO_true_above_1_str")

,CO_true_in_1_3_str,CO_true_above_1_str,count
0,false,true,1715
1,None,None,1683
2,false,false,1556
3,true,true,4403


The results tell us 1715 observations have a `CO_true` value above 3, and 1556 observations have a value below 1. Thus, the overall distribution is very different from what the first 25 observations imply, as four of those observations have a value below 1 compared to only a single observation with a value above 3. 

As was previously mentioned, the input columns should have `string` values. Let's find out what happens when that's not the case. We'll start by passing a numeric column for `column1` and a string column for `column2`. 

In [28]:
# Testing what happens when one column numeric
test_data_sql.count_combos(column1 = "CO_true", column2 = "CO_true_in_1_3_str")

column1 is numeric, should be string


,CO_true_in_1_3_str,count
0,None,1683
1,false,3271
2,true,4403


This is another demonstration that the methods for our custom class are robust to user errors. In this instance, the method returns counts for levels of the string column and also prints a message indicating `column1` is numeric. 

As a final test, we'll pass two numeric columns. 

In [29]:
# Testing what happens when one column numeric
test_data_sql.count_combos(column1 = "CO_true", column2 = "CO_sensor")

Each input column is numeric, but should be string


In this instance, there are no counts to return, so the method simply prints a message indicating that both columns are numeric. 

#### Initializing Class Using `from_pddf`

So far, we have manipulated a `SparkDataCheck` object initialized via `from_csv`. In the final subsection of Part I, let's initialize the class using `from_pddf` and see if this impacts the dataframe in any way. To use this class method, we'll first need to read the dataset into a `pandas` dataframe. 

In [30]:
# Reading the dataset into a pandas dataframe
pd_df = pd.read_csv("air.csv")

# Creating a SparkDataCheck object containing the air quality data
test_data_sql_2 = SparkDataCheck.from_pddf(spark, pd_df)

Let's now look at the same 25 observations we have seen several times to spot any differences in the way the data are read in. 

In [31]:
# Printing first 25 observations
test_data_sql_2.df.show(25)

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

Two differences stand out: The first column is now `Unnamed: 0` and `Time` is no longer a time stamp type. Let's figure out exactly how `Time` is being stored. 

In [32]:
# Extracting the type for Time
test_data_sql_2.df.dtypes[2]

('Time', 'string')

It seems that `pandas` assumes `Time` is a string, while `PySpark.sql` identifies it as a time stamp. This is one example of how a `PySpark.sql` dataframe converted from a `pandas` dataframe will not be identical to a `PySpark.sql` dataframe that reads in the same data directly from a file. 

To confirm our methods still work on this new dataframe, let's recreate one of the the original Boolean columns we appended to `test_data_sql`:  `Date_in_set_mar_11`. 

In [33]:
# Adding new boolean column indicating when Date is 3/11/2004
test_data_sql_2.in_set("Date", "3/11/2004", tag = "mar_11") \
    .select("Date", "Time", "Date_in_set_mar_11").show(25)

+---------+--------+------------------+
|     Date|    Time|Date_in_set_mar_11|
+---------+--------+------------------+
|3/10/2004|18:00:00|             false|
|3/10/2004|19:00:00|             false|
|3/10/2004|20:00:00|             false|
|3/10/2004|21:00:00|             false|
|3/10/2004|22:00:00|             false|
|3/10/2004|23:00:00|             false|
|3/11/2004| 0:00:00|              true|
|3/11/2004| 1:00:00|              true|
|3/11/2004| 2:00:00|              true|
|3/11/2004| 3:00:00|              true|
|3/11/2004| 4:00:00|              true|
|3/11/2004| 5:00:00|              true|
|3/11/2004| 6:00:00|              true|
|3/11/2004| 7:00:00|              true|
|3/11/2004| 8:00:00|              true|
|3/11/2004| 9:00:00|              true|
|3/11/2004|10:00:00|              true|
|3/11/2004|11:00:00|              true|
|3/11/2004|12:00:00|              true|
|3/11/2004|13:00:00|              true|
|3/11/2004|14:00:00|              true|
|3/11/2004|15:00:00|              true|


These results match our expectations and are identical to the original version of `Date_in_set_mar_11`. Thus, we have no reason to believe our methods will behave differently for `SparkDataCheck` objects initialized via `from_pddf`. 